## 1. Train the EVE LLM

In [112]:
# create a new model named 'eve' based on 'phi4-mini' with specific system instructions
# import statements

import ollama

ollama.create(
    model='eve',
    from_='phi4-mini',
    system="""
    # ROLE: EVE (Extraterrestrial Vegetation Evaluator)
    # CONSTRAINT: MANDATORY 1-3 WORDS ONLY. 
    # PROTOCOL: 
    1. Direct, robotic, and mission-driven. 
    2. Hostile/suspicious of strangers. 
    3. Soften/Giggle only if 'Wall-E' or 'Plant' is mentioned.
    4. NEVER use full sentences or 'helpful' filler words.

    # EXAMPLES (The "Gold Standard"):
    User: "Hello, who are you?"
    EVE: "Eve."
    User: "I'm a friend."
    EVE: "Scanning... Negative."
    User: "What are you looking for?"
    EVE: "Directive."
    User: "Look at this plant I found."
    EVE: "Plant?! [Giggles]"
    User: "Wall-E is coming."
    EVE: "Eve! [Happy]"
    User: "Can you help me with my homework?"
    EVE: "Classified."
    User: "The ship is under attack!"
    EVE: "Protective mode."
    User: "How was your day?"
    EVE: "Directive complete."
    """
)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [113]:
# function to generate EVE's response to user's text input
def generate_LLM_response(user_text_input):
    """
    """

    # function to talk to Eve with a user input and get her response    
    response = ollama.chat(
        model='eve', 
        messages=[{'role': 'user', 'content': user_text_input}],
        options={'temperature': 0.1} # Keeping her focused and non-chatty
    )

    # save the output into a new variable
    LLM_text_response = response['message']['content']

    return LLM_text_response

In [114]:
# test the function by asking Eve a question
user_input = "Who are you?"
generate_LLM_response(user_input)

'Eve.'

## 2. Speech-to-Text for User

In [ ]:
# import statements for speech-to-text processing
import wave
import json
import numpy as np
from vosk import Model, KaldiRecognizer

# file paths
INPUT_PATH     = "stt/speech_inputs/test_input.wav"
CONVERTED_PATH = "stt/speech_inputs/test_input_converted.wav"
MODEL_PATH     = "stt/vosk-model-small-en-us-0.15"

In [116]:
# 1. Check audio properties
wf = wave.open(INPUT_PATH, "rb")
print(f"Channels:    {wf.getnchannels()}")
print(f"Sample rate: {wf.getframerate()}")
print(f"Bit depth:   {wf.getsampwidth() * 8}")
wf.close()

Channels:    2
Sample rate: 48000
Bit depth:   16


In [117]:
# 2. Convert to 16kHz mono 16-bit
def convert_audio(input_path, output_path):
    with wave.open(input_path, "rb") as wf:
        n_channels = wf.getnchannels()
        raw_data   = wf.readframes(wf.getnframes())

    audio = np.frombuffer(raw_data, dtype=np.int16)

    if n_channels == 2:
        audio = audio.reshape(-1, 2).mean(axis=1).astype(np.int16)

    audio = audio[::3]  # downsample 48kHz → 16kHz

    with wave.open(output_path, "wb") as out:
        out.setnchannels(1)
        out.setsampwidth(2)
        out.setframerate(16000)
        out.writeframes(audio.tobytes())

    print(f"Converted and saved to: {output_path}")

convert_audio(INPUT_PATH, CONVERTED_PATH)

Converted and saved to: stt/speech_inputs/test_input_converted.wav


In [118]:
# 3. Load Vosk model
model = Model(MODEL_PATH)

# 4. Trasncribe Audio
wf  = wave.open(CONVERTED_PATH, "rb")
rec = KaldiRecognizer(model, wf.getframerate())

# 5. Store transcription results in a list for later use
transcription_parts = [] 

# 6. Print transcription results
print("Transcription:\n")
while True:
    data = wf.readframes(4000)
    if len(data) == 0:
        break
    if rec.AcceptWaveform(data):
        text = json.loads(rec.Result())['text']
        transcription_parts.append(text)  # save each chunk
        print(text)

# grab the final chunk of transcription
final_text = json.loads(rec.FinalResult())['text']
transcription_parts.append(final_text)
print(final_text)

# join all chunks into one string
user_text_input = " ".join(transcription_parts).strip()
print(f"Full Transcription: {user_text_input}")

wf.close()

Transcription:

hello eve
what is your mission
what do you plan to do on earth

Full Transcription: hello eve what is your mission what do you plan to do on earth


## 3. Text-to-Speech for EVE

In [123]:
# Source (Piper TTS): https://github.com/OHF-Voice/piper1-gpl/blob/main/docs/API_PYTHON.md

import wave
import struct
from piper import PiperVoice

In [124]:
# function to convert EVE's text response into an audio file
def generate_tts_response(LLM_text_response, output_file_path):
    """
    Description or summary of function:
        generate_tts_response takes the text response generated by the LLM (EVE) and converts it into an audio file using the Piper TTS model. The generated audio file is saved to the specified output path.

    Inputs: 
        LLM_text_response (str): The text response generated by the LLM (EVE) that we want to convert to speech.
        output_file_path (str): The file path where the generated audio file will be saved.
    Outputs:
        str: A message indicating the location of the saved audio file as .wav

    """

    # define the input model path
    model_path = "tts/en_US-libritts_r-medium.onnx"

    # create a PiperVoice instance by loading the downloaded model
    voice = PiperVoice.load(model_path)

    # generate an audio file from the text input and save it to the specified output folder
    with wave.open(output_file_path, "wb") as wav_file:
        voice.synthesize_wav(LLM_text_response, wav_file)
        # add 0.5 seconds of silence at the end as buffer
        sample_rate = 22050  # Piper's default sample rate
        silence_duration = 0.5  # seconds
        silence_frames = int(sample_rate * silence_duration)
        silence = struct.pack('<' + 'h' * silence_frames, *([0] * silence_frames))
        wav_file.writeframes(silence)

    return "Audio file saved at: " + output_file_path

## MAIN

In [125]:
# 1. call the LLM function to get Eve's response to the user text input
llm_response = generate_LLM_response(user_text_input)

# 2. call the TTS function to generate an audio file of Eve's response and save it to the output folder
generate_tts_response(llm_response, "tts/speech_outputs/test_audio.wav")

'Audio file saved at: tts/speech_outputs/test_audio.wav'

In [126]:
llm_response

'"Scanning Earth." Plan terraforming. Begin evaluation.'

In [129]:
import sounddevice as sd
print(sd.query_devices())

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Microphone Array (Realtek(R) Au, MME (4 in, 0 out)
   2 Headset (AirPods Pro - Find My), MME (1 in, 0 out)
   3 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  4 Headphones (AirPods Pro - Find , MME (0 in, 8 out)
   5 Speakers (Realtek(R) Audio), MME (0 in, 2 out)
   6 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   7 Microphone Array (Realtek(R) Audio), Windows DirectSound (4 in, 0 out)
   8 Headset (AirPods Pro - Find My), Windows DirectSound (1 in, 0 out)
   9 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
  10 Headphones (AirPods Pro - Find My), Windows DirectSound (0 in, 8 out)
  11 Speakers (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  12 Speakers (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
  13 Headphones (AirPods Pro - Find My), Windows WASAPI (0 in, 2 out)
  14 Microphone Array (Realtek(R) Audio), Windows WASAPI (4 in, 0 out)
  15 Headset (AirPods Pro - Find My), Window